In [1]:
force_rerun = True

# imports
import os
import sys

# code for enabling this notebook to work within cursor
coralme_dir = '/home/chris/zuniga/coralme/' #'../'
sys.path.insert(0, coralme_dir)

import subprocess
from cobra.io import load_json_model, write_sbml_model, save_json_model
from bs4 import BeautifulSoup
import pandas as pd
import json

def extract_json_from_window_data(html):
    start = html.find("window.data =")
    if start == -1:
        raise ValueError("Could not find 'window.data =' in the HTML.")

    start += len("window.data =")
    i = start
    brace_count = 0
    in_string = False
    escape = False

    # Skip whitespace to find the first {
    while html[i] in " \n\r\t":
        i += 1

    if html[i] != '{':
        raise ValueError("Expected '{' after 'window.data ='")

    json_start = i
    brace_count += 1
    i += 1

    # Parse until all braces are closed
    while i < len(html):
        char = html[i]

        if in_string:
            if escape:
                escape = False
            elif char == '\\':
                escape = True
            elif char == '"':
                in_string = False
        else:
            if char == '"':
                in_string = True
            elif char == '{':
                brace_count += 1
            elif char == '}':
                brace_count -= 1
                if brace_count == 0:
                    return html[json_start:i + 1]
        i += 1

    raise ValueError("Could not parse full JSON object from 'window.data ='")


In [3]:
# run memote on each, saving to json
base_dir = os.path.join(coralme_dir, 'species_files', 'Pseudomonas_files')
output_dfs = []
reverse = ['Presence of Gene Annotation', 'Presence of Metabolite Annotation', 'Presence of Reaction Annotation']
for f in os.listdir(os.path.join(base_dir, 'individual_species')):
    if 'Reference' in f: continue
    
    # look to see if memote solution already exists
    #M_json_path = os.path.join(base_dir, 'individual_species', f, 'inputs', 'model.json')
    M_json_path = os.path.join(base_dir, 'individual_species', f, 'inputs', 'model_fixed.json')
    M_xml_path = os.path.join(base_dir, 'individual_species', f, 'inputs', 'model.xml')
    report_path = os.path.join(base_dir, 'individual_species', f, 'outputs', 'memote_report.html')
    if force_rerun or (os.path.exists(M_json_path) and not os.path.exists(M_xml_path)):
        model = load_json_model(M_json_path)
        write_sbml_model(model, M_xml_path)

    # run memote
    if force_rerun or not os.path.exists(report_path):
        print(f+' running memote...')
        print(['memote', 'report', 'snapshot', '--filename', report_path, M_xml_path])
        fa = fo
        result = subprocess.run(['memote', 'report', 'snapshot', '--filename', report_path, M_xml_path], capture_output=True, text=True)

    # Load and extract results
    with open(report_path, encoding="utf-8") as file:
        html = file.read()
    
    json_str = extract_json_from_window_data(html)
    data = json.loads(json_str)
    
    # Summarize tests
    test_names = []
    test_results = []
    test_metrics = []
    
    # Loop through all tests in the memote data
    for test_id, test in data.get("tests", {}).items():
        name = test.get("title", test_id)
        result = test.get("result", None)
        metric = test.get("metric", None)
    
        # Normalize result (handle dicts like per-database results)
        if isinstance(result, dict):
            result = None
    
        # Normalize metric
        if not isinstance(metric, (int, float)):
            metric = None

        if name in reverse:
            metric = 1 - metric
    
        # Append to lists
        test_names.append(name)
        test_results.append(result)
        test_metrics.append(metric)
    
    # Optionally print the first few entries to verify
    output_df = pd.DataFrame(index = test_names)
    output_df[f+'_result'] = test_results
    output_df[f+'_metric'] = test_metrics
    output_dfs.append(output_df)

# concatenate all these output dataframes together
output_df = pd.concat(output_dfs, axis = 1)

CP008896 running memote...
['memote', 'report', 'snapshot', '--filename', '/home/chris/zuniga/coralme/species_files/Pseudomonas_files/individual_species/CP008896/outputs/memote_report.html', '/home/chris/zuniga/coralme/species_files/Pseudomonas_files/individual_species/CP008896/inputs/model.xml']


NameError: name 'fo' is not defined

In [6]:
'memote run /home/chris/zuniga/coralme/species_files/Pseudomonas_files/individual_species/CP008896/inputs/model.xml --log-level DEBUG'

'memote run /home/chris/zuniga/coralme/species_files/Pseudomonas_files/individual_species/CP008896/inputs/model.xml --log-level DEBUG'

In [87]:
# look into problem causing reactions
for f in os.listdir(os.path.join(base_dir, 'individual_species')):
    if 'Reference' in f: continue
    
    # look to see if memote solution already exists
    M_json_path = os.path.join(base_dir, 'individual_species', f, 'inputs', 'model.json')
    M_json_updated_path = os.path.join(base_dir, 'individual_species', f, 'inputs', 'model_fixed.json')
    #M_json_path = os.path.join(coralme_dir, 'species_files', 'iML1515.json')
    model = load_json_model(M_json_path)
    problematic_metabs = {}
    problematic_rxns = []
    for rxn in model.reactions:
        if 'EX_' in rxn.id or 'BIOMASS' in rxn.id: continue # these are imbalanced but obviously so
        input_eles = {}
        for metab in rxn.reactants:
            coef = rxn.get_coefficient(metab)
            for ele, ct in metab.elements.items():
                if ele not in input_eles:
                    input_eles.update({ele : 0})
                input_eles[ele] += abs(coef*ct)
        output_eles = {}
        for metab in rxn.products:
            coef = rxn.get_coefficient(metab)
            for ele, ct in metab.elements.items():
                if ele not in output_eles:
                    output_eles.update({ele : 0})
                output_eles[ele] += abs(coef*ct)
        if input_eles != output_eles:
            problematic_rxns.append(rxn.id)
            for metab in rxn.metabolites:
                if metab not in problematic_metabs:
                    problematic_metabs.update({metab : 0})
                problematic_metabs[metab] += 1
    print(f+': %age of problematic reactions = {:.2f}'.format(100*len(problematic_rxns) / len(model.reactions)))
    if len(problematic_rxns) < 10: print(problematic_rxns)
    print()

    # let's fix reactions
    if 'NITR_NO' in problematic_rxns:
        rxn = model.reactions.get_by_id('NITR_NO')
        rxn.add_metabolites({
            'h_c': 1.0,
        })
        save_json_model(model, M_json_updated_path)

CP008896: %age of problematic reactions = 0.06
['NITR_NO']

CP053697: %age of problematic reactions = 0.06
['NITR_NO']

CP026386: %age of problematic reactions = 0.06
['NITR_NO']

CP069317: %age of problematic reactions = 0.06
['NITR_NO']

AE004091.2: %age of problematic reactions = 0.05
['NITR_NO']

CP014784: %age of problematic reactions = 0.06
['NITR_NO']

CP073105: %age of problematic reactions = 0.06
['NITR_NO']

CP065866: %age of problematic reactions = 0.05
['NITR_NO']

CP061848: %age of problematic reactions = 0.06
['NITR_NO']

CP012831: %age of problematic reactions = 0.06
['NITR_NO']

CP008749.1: %age of problematic reactions = 0.05
['NITR_NO']

CP032419: %age of problematic reactions = 0.06
['NITR_NO']

CP038001: %age of problematic reactions = 0.06
['NITR_NO']

CP022560: %age of problematic reactions = 0.06
['NITR_NO']

CP076683: %age of problematic reactions = 0.06
['NITR_NO']

CP045416: %age of problematic reactions = 0.06
['NITR_NO']

AP022324: %age of problematic reacti

In [86]:
save_json_model

<function cobra.io.json.save_json_model(model: 'Model', filename: Union[str, pathlib.Path, IO], sort: bool = False, pretty: bool = False, **kwargs: Any) -> None>

In [83]:
# inspect a specific one
rxn = model.reactions.get_by_id('NITR_NO')
input_eles = {}
print(rxn.id, end = ': ')
print(rxn.reaction)
for metab in rxn.reactants:
    coef = rxn.get_coefficient(metab)
    print(str(coef)+' * '+metab.id, end = ': ')
    for ele, ct in metab.elements.items():
        if ele not in input_eles:
            input_eles.update({ele : 0})
        input_eles[ele] += abs(coef*ct)
        print(str(abs(coef*ct))+' '+ele, end = ', ')
    print()
output_eles = {}
for metab in rxn.products:
    coef = rxn.get_coefficient(metab)
    print(str(coef)+' * '+metab.id, end = ': ')
    for ele, ct in metab.elements.items():
        if ele not in output_eles:
            output_eles.update({ele : 0})
        output_eles[ele] += abs(coef*ct)
        print(str(abs(coef*ct))+' '+ele, end = ', ')
    print()
print('reactant total : %s' % input_eles)
print('product total : %s' % output_eles)

NITR_NO: focytc_c + h_c + no2_c --> ficytc_c + h2o_c + no_c
-1.0 * focytc_c: 42.0 C, 53.0 H, 1.0 Fe, 8.0 N, 6.0 O, 2.0 S, 
-1.0 * h_c: 1.0 H, 
-1.0 * no2_c: 1.0 N, 2.0 O, 
1.0 * ficytc_c: 42.0 C, 52.0 H, 1.0 Fe, 8.0 N, 6.0 O, 2.0 S, 
1.0 * h2o_c: 2.0 H, 1.0 O, 
1.0 * no_c: 1.0 N, 1.0 O, 
reactant total : {'C': 42.0, 'H': 54.0, 'Fe': 1.0, 'N': 9.0, 'O': 8.0, 'S': 2.0}
product total : {'C': 42.0, 'H': 54.0, 'Fe': 1.0, 'N': 9.0, 'O': 8.0, 'S': 2.0}


In [82]:
rxn

Reaction identifier,NITR_NO
Name,Nitrite reductase (NO forming)
Memory address,0x7f871029ecb0
Stoichiometry,focytc_c + h_c + no2_c --> ficytc_c + h2o_c + no_c Ferrocytochrome c C42H53FeN8O6S2 + H+ + Nitrite --> Ferricytochrome c C42H52FeN8O6S2 + H2O + Nitric oxide
GPR,
Lower bound,0.0
Upper bound,1000.0


In [65]:
# let's show the top problem causers
top_metabs = [k for k, _ in sorted(problematic_metabs.items(), key = lambda x : -x[1])]
for metab in top_metabs[0:30]:
    print(metab.id+' : n = '+str(problematic_metabs[metab]))

h2o_c : n = 3
h_c : n = 3
2fe2s_c : n = 2
PA_core_lipidA_c : n = 2
adp_c : n = 2
ala__L_c : n = 2
arg__L_c : n = 2
asn__L_c : n = 2
asp__L_c : n = 2
atp_c : n = 2
clpn160_c : n = 2
clpn161_c : n = 2
clpn180_c : n = 2
clpn181_c : n = 2
clpn_pa_17_0_cyc_c : n = 2
clpn_pa_19_0_cyc_c : n = 2
coa_c : n = 2
ctp_c : n = 2
cys__L_c : n = 2
datp_c : n = 2
dctp_c : n = 2
dgtp_c : n = 2
dttp_c : n = 2
gln__L_c : n = 2
glu__L_c : n = 2
gly_c : n = 2
glycogen_c : n = 2
gtp_c : n = 2
his__L_c : n = 2
ile__L_c : n = 2


In [66]:
# let's show the reactions
problematic_rxns

['BIOMASS_PA14_v27M', 'BIOMASS_PA14_v27M_ANA', 'NITR_NO']

1719